# Demo - Pipelines: dynamic tables keep data fresh, tasks add control

**Why we are doing this.** Two of Kenny's points:
- The team reprocesses entire datasets instead of only new or changed records.
- Dependent workflows, failures, retries and notifications are hard and mostly hand-built.

**Part A - Dynamic tables:** Bronze → Silver → Gold as plain SELECTs that stay fresh on their own.
**Part B - Tasks:** control when you need it: validate, retry, resume, notify.

| **Dynamic tables give you** | **Tasks add** |
|---|---|
| **Incremental:** only new, changed or deleted rows are processed. No CDC or MERGE code. | **Any step:** stored procedures, Python, dbt, API calls. |
| **Declarative:** a SELECT plus a freshness target. Snowflake schedules and orders refreshes. | **Control flow:** a task graph with conditions, e.g. validate before publishing. |
| **Dependency graph and lineage** built in, visible in the UI. | **Retries and resume** from the failed step. |
| **Refresh history and monitoring** built in. | **Notifications** on failure or success. |
| **Best for:** Gold tables and data products kept fresh for BI, semantic views and AI. | **Schedules** (cron) or **triggers** (new data arrives). |

*Dynamic tables describe what the data should look like. Tasks describe what steps should happen, and what to do when one fails.*

**For the Databricks engineers:** dynamic tables are closest to Delta Live Tables. Tasks are closest to Databricks Workflows or an ADF pipeline.

**Before you start:** 00_Ingestion must have been run, so Bronze has data.

**Labels:** **SETUP** (already built, read only) · **RUN DURING DEMO** · **UI STEP** · **SAY THIS**

### RUN DURING DEMO
Set the role, database, warehouse and schema. Run this first.

In [ ]:
%%sql -r ctx
USE ROLE ACCOUNTADMIN;
USE DATABASE SOMATUS_DEMO;
USE WAREHOUSE WH_TRANSFORM;
USE SCHEMA OPS;

# Part A - Dynamic tables: only new data is processed

### RUN DURING DEMO
**Silver: clean and de-duplicate members (latest record per member).** It includes DIALYSIS_MODALITY, the column AETNA's v2 file added in 00 Pass 2. This builds over 5 million rows in about 7 seconds.

In [ ]:
%%sql -r create_silver
CREATE OR REPLACE DYNAMIC TABLE SOMATUS_DEMO.SILVER.MEMBERS_CLEAN
  TARGET_LAG = '1 minute'          -- how fresh the data must be
  WAREHOUSE = WH_TRANSFORM
  REFRESH_MODE = INCREMENTAL       -- process only changed rows
AS
SELECT MEMBER_ID, CLIENT_ID, DOB, GENDER, CKD_STAGE, STATE, DIALYSIS_MODALITY, FILE_NAME, LOADED_AT
FROM SOMATUS_DEMO.BRONZE.MEMBERS_RAW
WHERE MEMBER_ID IS NOT NULL AND DOB IS NOT NULL AND LENGTH(STATE) = 2   -- business rules
QUALIFY ROW_NUMBER() OVER (PARTITION BY MEMBER_ID ORDER BY LOADED_AT DESC) = 1;  -- latest record wins

### SETUP - ALREADY DONE (read only, do not run)
**Silver: flatten the nested lab JSON into one row per test**

```sql
CREATE DYNAMIC TABLE SOMATUS_DEMO.SILVER.LAB_RESULTS
    TARGET_LAG = '1 minute'
    WAREHOUSE = WH_TRANSFORM
    REFRESH_MODE = INCREMENTAL
AS
SELECT RAW:lab_id::STRING AS LAB_ID, RAW:member_id::STRING AS MEMBER_ID, RAW:client_id::STRING AS CLIENT_ID,
       RAW:collected_at::TIMESTAMP_NTZ AS COLLECTED_AT, r.value:test::STRING AS TEST,
       r.value:value::FLOAT AS RESULT_VALUE, r.value:unit::STRING AS UNIT, FILE_NAME, LOADED_AT
FROM SOMATUS_DEMO.BRONZE.LABS_RAW, LATERAL FLATTEN(INPUT => RAW:panel.results) r;
```

### RUN DURING DEMO
**Gold: members by client, kidney disease stage and dialysis modality.** Built from Silver, in about 2 seconds.

In [ ]:
%%sql -r create_gold
CREATE OR REPLACE DYNAMIC TABLE SOMATUS_DEMO.GOLD.CLIENT_CKD_SUMMARY
  TARGET_LAG = '1 minute'          -- Gold stays within one minute of Silver
  WAREHOUSE = WH_TRANSFORM
  REFRESH_MODE = INCREMENTAL
AS
SELECT CLIENT_ID, CKD_STAGE, DIALYSIS_MODALITY, COUNT(*) AS MEMBERS
FROM SOMATUS_DEMO.SILVER.MEMBERS_CLEAN
GROUP BY 1, 2, 3;

### RUN DURING DEMO
The Gold result. Only AETNA has a modality (HOME_HD, IN_CENTER, PD), because only its v2 file sent the column. The other clients show blank.

In [ ]:
%%sql -r gold_view
SELECT * FROM SOMATUS_DEMO.GOLD.CLIENT_CKD_SUMMARY ORDER BY CLIENT_ID, CKD_STAGE, DIALYSIS_MODALITY;

> **SAY THIS:** There is no orchestration code here. You write the SELECT and say how fresh it must be. Snowflake works out the dependencies and refreshes only what changed. And notice DIALYSIS_MODALITY: the column AETNA added in Pass 2 now flows straight into Silver and Gold.

### RUN DURING DEMO
List the dynamic tables. Point out the target_lag and refresh_mode columns.

In [ ]:
%%sql -r dts
SHOW DYNAMIC TABLES IN DATABASE SOMATUS_DEMO
->> SELECT "schema_name" AS SCHEMA_NAME, "name" AS DYNAMIC_TABLE, "target_lag" AS TARGET_LAG,
           "refresh_mode" AS REFRESH_MODE, "scheduling_state" AS STATE, "rows" AS ROWS_
    FROM $1
    WHERE "name" IN ('MEMBERS_CLEAN', 'LAB_RESULTS', 'CLIENT_CKD_SUMMARY')
    ORDER BY CASE "schema_name" WHEN 'SILVER' THEN 1 ELSE 2 END, "name";

### UI STEP - do this in the screen, not in the notebook
**Show the pipeline graph**

1. In the left menu, click **Data**, then **Databases**.
2. Open **SOMATUS_DEMO > GOLD > Dynamic Tables** and click **CLIENT_CKD_SUMMARY**.
3. Click the **Graph** tab. You see Bronze MEMBERS_RAW feeding Silver MEMBERS_CLEAN feeding Gold CLIENT_CKD_SUMMARY.
4. Keep this tab open. Come back to **Refresh History** after Step 1.

> **SAY THIS:** Nobody drew this graph. Snowflake worked out the dependencies from the SELECT statements, and keeps each layer within its target lag.

## Step 1 - 100 new members arrive: watch them flow to Silver and Gold

### RUN DURING DEMO
**Before.** Rows in each layer now. Gold has one row per client, CKD stage and modality, so it is small. UnitedHealthcare is a new client, so it has 0 everywhere.

In [ ]:
%%sql -r layers_before
SELECT 'Bronze' AS LAYER, 'MEMBERS_RAW' AS TABLE_NAME, COUNT(*) AS TOTAL_ROWS,
       COUNT_IF(CLIENT_ID = 'UNITEDHEALTHCARE_DEMO') AS UHC_MEMBERS
FROM SOMATUS_DEMO.BRONZE.MEMBERS_RAW
UNION ALL
SELECT 'Silver', 'MEMBERS_CLEAN', COUNT(*), COUNT_IF(CLIENT_ID = 'UNITEDHEALTHCARE_DEMO')
FROM SOMATUS_DEMO.SILVER.MEMBERS_CLEAN
UNION ALL
SELECT 'Gold', 'CLIENT_CKD_SUMMARY', COUNT(*), COALESCE(SUM(IFF(CLIENT_ID = 'UNITEDHEALTHCARE_DEMO', MEMBERS, 0)), 0)
FROM SOMATUS_DEMO.GOLD.CLIENT_CKD_SUMMARY;

### RUN DURING DEMO
100 new members from a new client, UnitedHealthcare, land in Bronze. This stands in for a small new file.

In [ ]:
%%sql -r insert_new
INSERT INTO SOMATUS_DEMO.BRONZE.MEMBERS_RAW
  (MEMBER_ID, CLIENT_ID, DOB, GENDER, CKD_STAGE, STATE, FILE_NAME, LOADED_AT)
SELECT 'MUHC' || LPAD(SEQ4()::STRING, 5, '0'), 'UNITEDHEALTHCARE_DEMO', '1960-01-01', 'F', 'CKD5', 'VA',
       'uhc_new_members.csv', CURRENT_TIMESTAMP()
FROM TABLE(GENERATOR(ROWCOUNT => 100));

### RUN DURING DEMO
Refresh now instead of waiting up to one minute for the target lag. Refreshing Gold also refreshes Silver first, because Gold depends on it.

In [ ]:
%%sql -r refresh_now
ALTER DYNAMIC TABLE SOMATUS_DEMO.GOLD.CLIENT_CKD_SUMMARY REFRESH;

### RUN DURING DEMO
**After.** The same counts. Bronze and Silver each grew by 100. Gold grew by one row: UnitedHealthcare, CKD5, blank modality, 100 members.

In [ ]:
%%sql -r layers_after
SELECT 'Bronze' AS LAYER, 'MEMBERS_RAW' AS TABLE_NAME, COUNT(*) AS TOTAL_ROWS,
       COUNT_IF(CLIENT_ID = 'UNITEDHEALTHCARE_DEMO') AS UHC_MEMBERS
FROM SOMATUS_DEMO.BRONZE.MEMBERS_RAW
UNION ALL
SELECT 'Silver', 'MEMBERS_CLEAN', COUNT(*), COUNT_IF(CLIENT_ID = 'UNITEDHEALTHCARE_DEMO')
FROM SOMATUS_DEMO.SILVER.MEMBERS_CLEAN
UNION ALL
SELECT 'Gold', 'CLIENT_CKD_SUMMARY', COUNT(*), COALESCE(SUM(IFF(CLIENT_ID = 'UNITEDHEALTHCARE_DEMO', MEMBERS, 0)), 0)
FROM SOMATUS_DEMO.GOLD.CLIENT_CKD_SUMMARY;

### RUN DURING DEMO
**The proof it was incremental.** Silver inserted 100 rows and Gold inserted 1, not millions. Compare with the first build a few minutes ago: over 5 million rows. Refreshes where nothing changed (NO_DATA) are hidden; they cost nothing.

In [ ]:
%%sql -r refresh_proof
SELECT NAME, REFRESH_ACTION, REFRESH_TRIGGER,
       STATISTICS:numInsertedRows::INT AS ROWS_INSERTED,
       STATISTICS:numDeletedRows::INT  AS ROWS_DELETED,
       DATEDIFF('millisecond', REFRESH_START_TIME, REFRESH_END_TIME) AS MILLISECONDS,
       REFRESH_START_TIME
FROM TABLE(SOMATUS_DEMO.INFORMATION_SCHEMA.DYNAMIC_TABLE_REFRESH_HISTORY(
       DATA_TIMESTAMP_START => DATEADD('minute', -15, CURRENT_TIMESTAMP())))
WHERE NAME IN ('MEMBERS_CLEAN', 'CLIENT_CKD_SUMMARY') AND REFRESH_ACTION <> 'NO_DATA'
ORDER BY REFRESH_START_TIME DESC LIMIT 4;

> **SAY THIS:** Five million members in Silver, and the refresh touched only the 100 new ones. The same works when a client sends an updated record for an existing member: only that member is reprocessed. This is the end of reloading whole datasets.

## Dynamic tables or tasks?

| | Dynamic tables (Part A) | Tasks (Part B) |
|---|---|---|
| Answers | "What should this data look like, and how fresh?" | "What should happen, in what order, and what if it fails?" |
| Style | Declarative: you write the SELECT | Control flow: you write the steps |
| Dependencies | Worked out from the SQL | You declare them (AFTER) |
| Best at | Keeping Silver and Gold fresh, processing only changes | Validation gates, retries, resume from failure, emails |
| Databricks / ADF | Delta Live Tables | Workflows / ADF pipeline |

Most transformation work needs only dynamic tables. Use tasks when you need control. They work together: in Part B, a task refreshes a dynamic table only after validation passes.

> **SAY THIS:** Dynamic tables orchestrate the data. Tasks orchestrate the process. You've seen dynamic tables keep data fresh. Now: what if you need control?

# Part B - Tasks: failure, retry, alert and recovery

**What we will prove.**
1. A chain of dependent tasks runs in order: validate, then transform, then publish.
2. When a step fails, Snowflake retries it automatically.
3. When it still fails, a final step (the finalizer) emails the team, and nothing downstream runs with bad data.
4. After the fix, the chain resumes from the failed step. Steps that already succeeded are not rerun.

### SETUP - ALREADY DONE (read only, do not run)
**The gated Gold table**

```sql
-- Published claims for reports and clients. TARGET_LAG = 'DOWNSTREAM' and nothing reads from it,
-- so Snowflake never refreshes it on its own. Only the pipeline's transform step refreshes it,
-- and that step runs only after validation passes. This is the gate.
CREATE DYNAMIC TABLE SOMATUS_DEMO.GOLD.CLAIMS_PUBLISHED
    TARGET_LAG = 'DOWNSTREAM'
    WAREHOUSE = WH_TRANSFORM
    REFRESH_MODE = INCREMENTAL
AS
SELECT CLAIM_ID, MEMBER_ID, CLIENT_ID, SERVICE_DATE, CPT_CODE,
       PAID_AMOUNT, CLAIM_STATUS, FILE_NAME, LOADED_AT
FROM SOMATUS_DEMO.SILVER.CLIENT_CLAIMS;
```

### SETUP - ALREADY DONE (read only, do not run)
**The task graph**

```sql
 
-- Root: starts the run. Retry the whole graph once automatically on failure.
## T_ROOT is the start button of the claims pipeline. Its own work is tiny: it writes one line, "START OK, run started", into OPS.PIPELINE_LOG.
CREATE TASK OPS.T_ROOT WAREHOUSE=WH_INGEST TASK_AUTO_RETRY_ATTEMPTS = 1
  AS CALL OPS.SP_START_RUN();
  
-- Children run only after their parent succeeds
## T_VALIDATE is the quality gate of the pipeline. It checks the new claims, and if anything is wrong it fails on purpose, so nothing after it runs.
CREATE TASK OPS.T_VALIDATE  WAREHOUSE=WH_INGEST    AFTER OPS.T_ROOT     AS CALL OPS.SP_VALIDATE_BATCH();

## T_TRANSFORM is the step that actually puts the claims into Gold, and it runs only if validation passed.
CREATE TASK OPS.T_TRANSFORM WAREHOUSE=WH_TRANSFORM AFTER OPS.T_VALIDATE
  AS ALTER DYNAMIC TABLE GOLD.CLAIMS_PUBLISHED REFRESH;   -- publish only after validation passes

## T_PUBLISH moves no data. It only writes a "done" record into OPS.PIPELINE_LOG  
CREATE TASK OPS.T_PUBLISH   WAREHOUSE=WH_TRANSFORM AFTER OPS.T_TRANSFORM
  AS INSERT INTO OPS.PIPELINE_LOG (STEP, STATUS, DETAIL) SELECT 'PUBLISH','OK','CLAIMS_PUBLISHED rows=' || COUNT(*) FROM GOLD.CLAIMS_PUBLISHED;
  
-- Finalizer: always runs at the end, success or failure, and emails the outcome
## T_NOTIFY is the finalizer: the task that always runs once at the end of a run, whether it succeeded or failed. Its procedure, SP_NOTIFY_RUN, checks how the run went and reports it.
CREATE TASK OPS.T_NOTIFY WAREHOUSE=WH_INGEST FINALIZE = OPS.T_ROOT AS CALL OPS.SP_NOTIFY_RUN();

-- Production schedule (not used in the demo):
-- ALTER TASK T_ROOT SET SCHEDULE = 'USING CRON 0 5 * * * America/New_York';
```

### SETUP - ALREADY DONE (read only, do not run)
**The SLA alert**

```sql
CREATE ALERT OPS.ALERT_GOLD_SLA WAREHOUSE=WH_INGEST SCHEDULE='60 MINUTE'
IF (EXISTS (SELECT 1 FROM (SELECT MAX(LOGGED_AT) LAST_PUBLISH FROM OPS.PIPELINE_LOG WHERE STEP='PUBLISH' AND STATUS='OK')
            WHERE LAST_PUBLISH IS NULL OR LAST_PUBLISH < DATEADD('minute', -5, CURRENT_TIMESTAMP())))
THEN CALL SYSTEM$SEND_EMAIL('SOMATUS_DEMO_EMAIL', 'mayank.p.gupta@snowflake.com',
       '[Somatus demo] SLA MISSED - Gold claims not published', '...');
```

**How we make it fail on purpose.** The validation procedure reads a switch in the table OPS.DEMO_CONTROL. When FAIL_VALIDATE is 'TRUE', validation raises an error. This stands in for a real problem such as a client file with a wrong total.

### UI STEP - do this in the screen, not in the notebook
**Show the graph visually**

1. In the left menu, click **Data**, then **Databases**.
2. Open **SOMATUS_DEMO > OPS > Tasks** and click **T_ROOT**.
3. Click the **Graph** tab. You see T_ROOT, then T_VALIDATE, then T_TRANSFORM, then T_PUBLISH, with T_NOTIFY as the finalizer.
4. Keep this tab open. You will come back to it in Step 2 to see the failed step in red.

## Step 1 - A new batch of claims arrives

### RUN DURING DEMO
Insert 500 new claims into Bronze. They are now waiting for the pipeline to validate and publish them.

In [ ]:
%%sql -r new_batch
INSERT INTO SOMATUS_DEMO.BRONZE.CLAIMS_RAW
  (CLAIM_ID, MEMBER_ID, CLIENT_ID, SERVICE_DATE, CPT_CODE, PAID_AMOUNT, CLAIM_STATUS, FILE_NAME, LOADED_AT)
SELECT 'LIVE_' || UUID_STRING(), 'M00000001', 'HUMANA_DEMO', '2026-10-21', '90935', 450, 'PAID',
       'live_batch', CURRENT_TIMESTAMP()
FROM TABLE(GENERATOR(ROWCOUNT => 500));

### RUN DURING DEMO
You should see ARRIVED_IN_BRONZE = 500 and PUBLISHED_TO_GOLD = 0.

In [ ]:
%%sql -r pending
SELECT (SELECT COUNT(*) FROM SOMATUS_DEMO.BRONZE.CLAIMS_RAW WHERE FILE_NAME = 'live_batch') AS ARRIVED_IN_BRONZE,
       (SELECT COUNT(*) FROM SOMATUS_DEMO.GOLD.CLAIMS_PUBLISHED WHERE FILE_NAME = 'live_batch') AS PUBLISHED_TO_GOLD;

## Step 2 - Break it and run the pipeline

### RUN DURING DEMO
Turn on the failure switch, clear the log, and start the pipeline.

In [ ]:
%%sql -r break
UPDATE DEMO_CONTROL SET VAL = 'TRUE' WHERE KEY = 'FAIL_VALIDATE';
TRUNCATE TABLE PIPELINE_LOG;
EXECUTE TASK T_ROOT;

### RUN DURING DEMO
Run this about 30 seconds later, and again after another 30 seconds. You will see T_VALIDATE **FAILED**, then attempt 2 (the automatic retry) also **FAILED**, and T_NOTIFY **SUCCEEDED**. T_TRANSFORM and T_PUBLISH never ran.

In [ ]:
%%sql -r history
SELECT NAME, STATE, ATTEMPT_NUMBER, LEFT(ERROR_MESSAGE, 100) AS ERROR, SCHEDULED_TIME
FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(SCHEDULED_TIME_RANGE_START => DATEADD('minute', -10, CURRENT_TIMESTAMP())))
WHERE NAME IN ('T_ROOT','T_VALIDATE','T_TRANSFORM','T_PUBLISH','T_NOTIFY')
ORDER BY SCHEDULED_TIME, NAME;

### RUN DURING DEMO
The pipeline's own log shows where it stopped.

In [ ]:
%%sql -r log_failed
SELECT * FROM PIPELINE_LOG ORDER BY LOGGED_AT;

### UI STEP - do this in the screen, not in the notebook
**Show the failure in the screen and the email**

1. Go back to the **T_ROOT > Graph** tab and refresh. T_VALIDATE is shown in red. Click it to see the error message.
2. Click the **Run History** tab to see both attempts.
3. Open your email inbox (mayank.p.gupta@snowflake.com) and show the failure email sent by the finalizer.

> **SAY THIS:** The failure was caught, retried, and reported, and Gold was never touched with bad data.

## Step 3 - Fix it and resume from the failed step

### RUN DURING DEMO
Turn the failure switch off (the fix), then resume the last run. RETRY LAST restarts from T_VALIDATE. T_ROOT, which already succeeded, is not rerun.

In [ ]:
%%sql -r fix_retry
UPDATE DEMO_CONTROL SET VAL = 'FALSE' WHERE KEY = 'FAIL_VALIDATE';
EXECUTE TASK T_ROOT RETRY LAST;

### RUN DURING DEMO
Run this about 30 seconds later. The latest attempt succeeds through T_VALIDATE, T_TRANSFORM and T_PUBLISH. A success email arrives.

In [ ]:
%%sql -r history_after
SELECT NAME, STATE, ATTEMPT_NUMBER, LEFT(ERROR_MESSAGE, 100) AS ERROR, SCHEDULED_TIME
FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(SCHEDULED_TIME_RANGE_START => DATEADD('minute', -15, CURRENT_TIMESTAMP())))
WHERE NAME IN ('T_ROOT','T_VALIDATE','T_TRANSFORM','T_PUBLISH','T_NOTIFY')
ORDER BY SCHEDULED_TIME, NAME;

### RUN DURING DEMO
The log now shows PUBLISH OK.

In [ ]:
%%sql -r log_ok
SELECT * FROM PIPELINE_LOG ORDER BY LOGGED_AT;

### RUN DURING DEMO
You should see 500 and 500. The claims were published once, only after validation passed, even though the first run failed and was retried.

In [ ]:
%%sql -r published
SELECT (SELECT COUNT(*) FROM SOMATUS_DEMO.BRONZE.CLAIMS_RAW WHERE FILE_NAME = 'live_batch') AS ARRIVED_IN_BRONZE,
       (SELECT COUNT(*) FROM SOMATUS_DEMO.GOLD.CLAIMS_PUBLISHED WHERE FILE_NAME = 'live_batch') AS PUBLISHED_TO_GOLD;

> **SAY THIS:** Recovery is one command, from the exact point of failure, with no duplicate processing.

## Step 4 (talk only) - The SLA alert

> **SAY THIS (do not run in the 45-minute cut):** Failures are one problem. The other is a pipeline that simply runs late. A separate alert checks every hour whether Gold was published on time, and emails the team if it was not. That is the setup cell at the top of this notebook.

### IF ASKED - run only if someone wants to see it
Run the SLA check now. It emails only if Gold was not published in the last 5 minutes (a short window for the demo). Right after Step 3 it will not fire; it fires about 5 minutes later.

In [ ]:
%%sql -r alert
EXECUTE ALERT ALERT_GOLD_SLA;

### IF ASKED
Show the alert's history. STATE is TRIGGERED when the email was sent, CONDITION_FALSE when everything was on time.

In [ ]:
%%sql -r alert_history
SELECT NAME, STATE, SCHEDULED_TIME, SQL_ERROR_MESSAGE
FROM TABLE(INFORMATION_SCHEMA.ALERT_HISTORY(SCHEDULED_TIME_RANGE_START => DATEADD('hour', -1, CURRENT_TIMESTAMP())))
ORDER BY SCHEDULED_TIME DESC;

## What we just proved
1. **Only new data is processed.** 100 new members in Bronze: Silver inserted 100 rows and Gold 1, out of more than 5 million members. No full reload.
2. **No orchestration code for the data.** Bronze to Silver to Gold is plain SELECT statements with a target lag. Snowflake built the dependency graph.
3. **Inserts and updates both flow through.** Only the changed rows are reprocessed.
4. **Tasks add control.** Validate, then transform, then publish, in order.
5. **Failures are caught and retried automatically.** The finalizer emailed the team.
6. **Bad data never reached Gold.** Published stayed at 0 until the fix.
7. **Recovery is one command.** RETRY LAST resumed from the failed step; 500 claims were published exactly once.

> **SAY THIS:** Dynamic tables keep the data fresh, tasks keep the process safe, and both run on the same platform with no extra tooling.